*Figure notebook of our analysis (IDIBAPS cFos brains). It reads the project settings from the environment variable `CFOS_MAPPER_SETTINGS` (or `./settings.json`); set it before running:* `%env CFOS_MAPPER_SETTINGS=/path/to/settings.json`

# Step 6 - figures explaining the method (for presentations)

Kernel: **cellfinder-latest (1.11)**

All brain pictures are **coronal** (dorsal up, left–right as imaged), drawn in true micrometers.

| figure | shows |
|---|---|
| M1 | the pipeline, and the annotate → train → test loop |
| M2 | coronal sections of the brains with the test regions used for annotation |
| M3 | step 1: candidates (bright, cell-sized spots) |
| M4 | the manual annotations per region |
| M5 | what the network sees: small 3D cubes around cells and non-cells |
| M6 | training curves |
| M7 | brightness differences between brains, and the correction |
| M8 | cellfinder's pretrained network vs ours, on annotations NOT used for training |
| M9 | the same region classified by the pretrained network and by ours |

Figures are saved as PDF (editable in Illustrator) and PNG (slides) in `figures/method_<network>/`.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Rectangle
import tifffile
import cfos_mapper as cf
from cfos_mapper.figures import style as fs

fs.apply_style()
settings = cf.load_settings()
results = Path(settings['results_folder'])
networks = Path(settings['networks_folder'])
regions_file = Path(settings['regions_file'])
regions = {b: r for b, r in json.load(open(regions_file)).items() if not b.startswith('_')}
brains = sorted(regions)

network = 'network_v2'                       # our trained network
example = ('O16', 'regionA_cortex')          # region used for the example pictures (M3, M9)
slab = (225, 255)                            # coronal slab of the region shown in M3, M9 (rows along y, ~44 µm)

out = results.parent / 'figures' / f'method_{network}'

# Axes of the data: z = dorsal (plane 0) -> ventral, y = anterior -> posterior, x = left-right.
# Coronal view = z (up-down, dorsal up) x x (left-right), drawn in micrometers.

def region_label(brain, region):
    return f'{brain} {region[6]}'   # e.g. 'O15 A'

## M1. Pipeline and training loop

In [ ]:
def flow(steps, name, loop_label=None):
    fig, ax = plt.subplots(figsize=(3.3 * len(steps), 3.0 if loop_label else 2.4))
    ax.axis('off')
    ax.set_xlim(0, len(steps))
    ax.set_ylim(-0.4 if loop_label else 0, 1)
    for i, (title, text) in enumerate(steps):
        ax.add_patch(FancyBboxPatch((i + 0.06, 0.12), 0.78, 0.76, boxstyle='round,pad=0.02,rounding_size=0.06',
                                    facecolor='white', edgecolor=fs.AXIS, linewidth=1))
        ax.text(i + 0.45, 0.67, title, ha='center', va='center', fontsize=13, fontweight='bold')
        ax.text(i + 0.45, 0.37, text, ha='center', va='center', fontsize=10, color=fs.INK_2)
        if i < len(steps) - 1:
            ax.annotate('', xy=(i + 1.05, 0.5), xytext=(i + 0.87, 0.5),
                        arrowprops=dict(arrowstyle='-|>', color=fs.MUTED, lw=1.5))
    if loop_label:
        # back from the last box to the first one, underneath the boxes
        first, last, low = 0.45, len(steps) - 0.55, -0.15
        ax.plot([last, last, first], [0.09, low, low], color=fs.MUTED, lw=1.5, solid_joinstyle='round')
        ax.annotate('', xy=(first, 0.09), xytext=(first, low),
                    arrowprops=dict(arrowstyle='-|>', color=fs.MUTED, lw=1.5, shrinkA=0, shrinkB=0))
        ax.text((first + last) / 2, low - 0.1, loop_label, ha='center', va='top', fontsize=10, color=fs.INK_2)
    fs.save(fig, out, name)

flow([('Whole brain', 'light-sheet images\ncFos + autofluorescence\n(opened with pymif)'),
      ('1. Find candidates', 'bright, cell-sized spots\n(cellfinder detection)'),
      ('2. Classify', 'our network decides:\ncell or not a cell'),
      ('3. Results', 'cell positions\ndensity maps')], 'M1a_pipeline')

flow([('Annotate', 'mark cells and non-cells\nin test regions (napari)'),
      ('Train', 'start from cellfinder\'s\npretrained network'),
      ('Test', 'on 20% of the annotations\nkept out of training'),
      ('Review', 'check where network\nand annotator disagree')],
     'M1b_training_loop', loop_label='repeat with more annotations')

## M2. Brains and test regions

In [ ]:
level = 2
names = list(next(iter(regions.values())))   # regionA_cortex, regionB_central
fig, axes = plt.subplots(len(names), len(brains), figsize=(3.6 * len(brains), 2.7 * len(names) + 0.6), squeeze=False)
for col, brain in enumerate(brains):
    data = cf.open_brain(brain, settings)
    full_um, low_um = np.array(cf.voxel_size(data)), np.array(cf.voxel_size(data, level))
    cfos = cf.get_channel(data, settings['signal_channel'], level)
    for row, name in enumerate(names):
        r = regions[brain][name]
        y_low = int(np.mean(r['y']) * full_um[1] / low_um[1])     # coronal section through the region
        section = np.asarray(cfos[:, y_low, :])
        extent = [0, section.shape[1] * low_um[2] / 1000, section.shape[0] * low_um[0] / 1000, 0]
        ax = axes[row, col]
        ax.imshow(section, cmap='gray', vmin=np.percentile(section, 5), vmax=np.percentile(section, 99.7),
                  extent=extent)
        x0, z0 = r['x'][0] * full_um[2] / 1000, r['z'][0] * full_um[0] / 1000
        w, h = (r['x'][1] - r['x'][0]) * full_um[2] / 1000, (r['z'][1] - r['z'][0]) * full_um[0] / 1000
        ax.add_patch(Rectangle((x0, z0), w, h, fill=False, edgecolor=fs.YELLOW, linewidth=1.5))
        ax.text(x0 + w + 0.15, z0 + h / 2, name[6], color=fs.YELLOW, va='center', fontsize=11, fontweight='bold')
        fs.image_axes(ax)
        if row == 0:
            ax.set_title(brain)
    axes[0, 0].set_ylabel('region A', color=fs.INK_2)
    axes[1, 0].set_ylabel('region B', color=fs.INK_2)
fs.scale_bar(axes[-1, -1], 2, 1, label='2 mm')
fs.title(fig, 'Brains (coronal sections of cFos, dorsal up) and test regions: A = cortex, B = centre')
fs.save(fig, out, 'M2_brains_and_test_regions')

## M3. Step 1: candidates

In [ ]:
signal, background, size, candidates = cf.prepare_region(*example, settings, regions_file)
y0, y1 = slab
projection = signal[:, y0:y1, :].max(1)                     # coronal: z (dorsal up) x x
extent_um = [0, projection.shape[1] * size[2], projection.shape[0] * size[0], 0]
contrast = dict(cmap='gray', vmin=np.percentile(signal, 30), vmax=np.percentile(signal, 99.9), extent=extent_um)
shown = candidates[(candidates.y >= y0) & (candidates.y < y1)]

def coronal_points(table):
    """x, z of points in micrometers (for drawing on the coronal view)."""
    return table.x * size[2], table.z * size[0]

fig, axes = plt.subplots(1, 2, figsize=(12, 7.4))
for ax, with_candidates in zip(axes, [False, True]):
    ax.imshow(projection, **contrast)
    if with_candidates:
        ax.scatter(*coronal_points(shown), s=90, facecolors='none', edgecolors='#f0efec', linewidths=1)
    fs.image_axes(ax)
    fs.scale_bar(ax, 100, 1)
axes[0].set_title('cFos')
axes[1].set_title(f'candidates ({len(shown)} in this slab)')
fs.title(fig, f'Step 1: bright, cell-sized spots ({region_label(*example)}, coronal slab of '
              f'{(y1 - y0) * size[1]:.0f} µm, dorsal up)')
fs.save(fig, out, 'M3_candidates')

## M4. Manual annotations

In [ ]:
rows = []
for brain in brains:
    for region in regions[brain]:
        a = cf.load_annotations(results / brain / 'test_regions' / f'{region}_annotations.csv')
        rows.append(dict(region=region_label(brain, region), cells=int((a.label == 'cell').sum()),
                         non_cells=int((a.label == 'not_cell').sum())))
annotations = pd.DataFrame(rows)

x = np.arange(len(annotations))
fig, ax = plt.subplots(figsize=(1.2 * len(annotations) + 2, 4.2))
for shift, column, color, label in [(-0.2, 'cells', fs.ORANGE, 'cells'), (0.2, 'non_cells', fs.BLUE, 'non-cells')]:
    bars = ax.bar(x + shift, annotations[column], width=0.38, color=color, label=label)
    fs.bar_labels(ax, bars)
ax.set_xticks(x, annotations.region)
ax.tick_params(axis='x', length=0)
ax.set_ylabel('annotations')
ax.set_title(f'Manual annotations: {annotations.cells.sum()} cells, {annotations.non_cells.sum()} non-cells')
ax.legend(loc='upper right')
ax.margins(y=0.12)
fs.save(fig, out, 'M4_annotations')

## M5. What the network sees
Each candidate is judged from a small 3D cube of image around it (here: cFos, projected along z), after correcting the brightness of the brain.

In [ ]:
n_examples = 8
rng = np.random.default_rng(1)
cube_root = networks / network / 'training_cubes'
# a cube is 20 x 50 x 50 voxels of 5 x 1 x 1 µm (z, y, x): 100 µm deep, 50 µm wide
cube_extent = [0, 50, 100, 0]

fig, axes = plt.subplots(2, n_examples, figsize=(1.3 * n_examples, 6.4))
for row, (kind, label, color) in enumerate([('cells', 'cells', fs.ORANGE), ('non_cells', 'non-cells', fs.BLUE)]):
    files = sorted(cube_root.glob(f'*/{kind}/*Ch0.tif'))
    for ax, f in zip(axes[row], rng.choice(files, min(n_examples, len(files)), replace=False)):
        cube = tifffile.imread(f).max(1)            # coronal view: z (dorsal up) x x
        ax.imshow(cube, cmap='gray', vmin=np.percentile(cube, 1), vmax=np.percentile(cube, 99.5),
                  extent=cube_extent)
        fs.image_axes(ax)
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_edgecolor(color)
            spine.set_linewidth(2)
    axes[row, 0].set_ylabel(label, color=fs.INK, fontsize=12, fontweight='bold')
fs.title(fig, 'What the network sees: cubes around annotated cells and non-cells '
              '(coronal view, 50 µm wide × 100 µm deep)')
fs.save(fig, out, 'M5_network_input')

## M6. Training

In [ ]:
log = pd.read_csv(networks / network / 'training.csv')
log['epoch'] += 1
best = log.loc[log.val_loss.idxmin()]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
for ax, metric, title in [(axes[0], 'accuracy', 'Accuracy (higher is better)'), (axes[1], 'loss', 'Error (lower is better)')]:
    for column, color, label in [(metric, fs.BLUE, 'training cubes'), (f'val_{metric}', fs.ORANGE, 'validation cubes')]:
        ax.plot(log.epoch, log[column], color=color, label=label, linewidth=1.6)
    ax.axvline(best.epoch, color=fs.MUTED, linestyle='--', linewidth=1)
    ax.annotate(f'kept: epoch {int(best.epoch)}', (best.epoch, ax.get_ylim()[1]), xytext=(4, -12),
                textcoords='offset points', color=fs.INK_2, fontsize=9)
    ax.set_xlabel('epoch')
    ax.set_title(title)
axes[0].legend(loc='lower right')
fs.title(fig, f'Training {network} (starting from cellfinder\'s pretrained network)')
fs.save(fig, out, 'M6_training')

## M7. Brightness differences between brains
Median brightness of the tissue, before correction. Every brain is scaled to the same level (dashed line) before the network looks at it.

In [ ]:
factors = {brain: cf.brightness_factors(brain, settings) for brain in brains}
reference = settings['brightness_reference']

fig, axes = plt.subplots(1, 2, figsize=(4 + 1.8 * len(brains), 4))
for ax, channel, title in [(axes[0], 'signal', 'cFos'), (axes[1], 'background', 'autofluorescence')]:
    values = [factors[b]['tissue_median'][channel] for b in brains]
    bars = ax.bar(brains, values, color=fs.BLUE if channel == 'signal' else fs.AQUA, width=0.55)
    fs.bar_labels(ax, bars)
    ax.axhline(reference[channel], color=fs.INK_2, linestyle='--', linewidth=1)
    # label at the right end of the dashed line (x in axes fraction, y in data)
    ax.text(0.99, reference[channel], 'all brains after correction', transform=ax.get_yaxis_transform(),
            ha='right', va='bottom', color=fs.INK_2, fontsize=9)
    ax.set_title(f'{title}: median tissue brightness')
    ax.tick_params(axis='x', length=0)
    ax.set_ylim(0, max(max(values), reference[channel]) * 1.18)
fig.tight_layout()
fs.save(fig, out, 'M7_brightness')

## M8. Pretrained network vs ours
Both networks are tested on the same annotations, the ones **not used** to train ours. Random spots in plain tissue (> 10 µm from any candidate) should never be called a cell. Computes predictions that are missing (about 1 minute per region, once).

In [ ]:
rows = []
for brain in brains:
    for region in regions[brain]:
        annotated = cf.load_annotations(results / brain / 'test_regions' / f'{region}_annotations.csv')
        test = annotated[~cf.used_in_training(annotated, networks / network, brain, region)]
        row = dict(region=region_label(brain, region))
        for name, key in [('pretrained', 'pretrained'), (network, 'ours')]:
            p = cf.get_predictions(brain, region, name, settings, regions_file)
            plain = cf.far_from_candidates(p['control'], p['candidates'], p['size'])
            a = cf.compare_with_annotations(p['calls'], test)
            n_cells = a['cells_found'] + a['cells_missed']
            n_not = a['false_positives'] + a['correctly_rejected']
            row[f'{key}: cells found (%)'] = 100 * a['cells_found'] / n_cells if n_cells else np.nan
            row[f'{key}: non-cells called cell (%)'] = 100 * a['false_positives'] / n_not if n_not else np.nan
            row[f'{key}: plain tissue called cell (%)'] = 100 * plain.is_cell.mean()
            row['test cells'], row['test non-cells'] = n_cells, n_not
        rows.append(row)
comparison = pd.DataFrame(rows)
out.mkdir(parents=True, exist_ok=True)
comparison.round(1).to_csv(out / 'M8_comparison.csv', index=False)

panels = [('cells found (%)', 'Annotated cells found (higher is better)'),
          ('non-cells called cell (%)', 'Annotated non-cells called cell (lower is better)'),
          ('plain tissue called cell (%)', 'Plain tissue called cell (lower is better)')]
x = np.arange(len(comparison))
fig, axes = plt.subplots(len(panels), 1, figsize=(1.1 * len(comparison) + 3, 3.2 * len(panels) + 0.8), sharex=True)
for ax, (measure, title) in zip(axes, panels):
    for shift, key, color, label in [(-0.2, 'pretrained', fs.PRETRAINED, 'cellfinder pretrained'),
                                     (0.2, 'ours', fs.OURS, f'ours ({network})')]:
        bars = ax.bar(x + shift, comparison[f'{key}: {measure}'], width=0.38, color=color, label=label)
        fs.bar_labels(ax, bars)
    ax.set_title(title)
    ax.set_ylim(0, 112)
    ax.set_ylabel('%')
# under each region: how many test annotations the percentages are based on
axes[-1].set_xticks(x, [f'{r.region}\n{r["test cells"]} / {r["test non-cells"]}' for _, r in comparison.iterrows()])
axes[-1].set_xlabel('test annotations: cells / non-cells (few annotations = uncertain %)')
axes[-1].tick_params(axis='x', length=0)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper left', ncols=2, bbox_to_anchor=(0.01, 0.995))
fig.tight_layout(rect=(0, 0, 1, 0.96))
fs.save(fig, out, 'M8_pretrained_vs_ours')
comparison.round(1)

## M9. The same region, classified by both networks

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 7.6))
for ax, (name, title) in zip(axes, [('pretrained', 'cellfinder pretrained'), (network, f'ours ({network})')]):
    calls = cf.get_predictions(*example, name, settings, regions_file)['calls']
    shown = calls[(calls.y >= y0) & (calls.y < y1)]
    ax.imshow(projection, **contrast)
    for is_cell, color in [(False, fs.NOT_CELL), (True, fs.CELL)]:
        ax.scatter(*coronal_points(shown[shown.is_cell == is_cell]), s=90, facecolors='none',
                   edgecolors=color, linewidths=1.1)
    fs.image_axes(ax)
    fs.scale_bar(ax, 100, 1)
    ax.set_title(f'{title}: {int(shown.is_cell.sum())} cells, {int((~shown.is_cell).sum())} not cells')
fs.title(fig, f'{region_label(*example)}, coronal slab of {(y1 - y0) * size[1]:.0f} µm (dorsal up):  '
              f'orange = cell,  blue = not a cell')
fs.save(fig, out, 'M9_example_both_networks')